# CS412: Bisaya Speech Affect Recognition - Machine Learning Experiments
**Course:** CS412 – Affective Computing Research Mini-Project  
**Cohort:** Group 7 (4th-Year Computer Science)  
**Deliverable:** 10-Minute Live Technical Checking – Part C: Machine Learning and Results  

---

## 1. Experimental Overview & Setup
This notebook demonstrates the Speech Emotion Recognition (SER) training procedure and evaluation using **108 acoustic & prosodic speech features** across **40 spontaneous Bisaya utterances**.

### Research Questions Addressed:
* **RQ4:** To what extent can machine learning models classify students' self-reported affective states using acoustic and prosodic features?
* **RQ5:** Which machine learning model provides the best performance for the collected Bisaya post-class speech dataset?

### Strict Leakage Control Protocol:
* Self-report ratings and educational context are strictly excluded from predictor matrices.
* Missing pitch imputation (`SimpleImputer(strategy='median')`) and standardization (`StandardScaler()`) are fit **strictly inside each training fold** via `sklearn.pipeline.Pipeline`.
* Validation Scheme: **Stratified 5-Fold Cross-Validation** (preserving class proportions).

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

sys.path.append(os.path.abspath('.'))
from src.multimodal_dataset import predictor_columns
print('[OK] Environment configured and libraries loaded!')

## 2. Dataset Loading & Feature Space Verification
Load `cleaned_multimodal_dataset.csv` and extract the 108 acoustic predictors.

In [ ]:
df = pd.read_csv('datasets/multimodal/cleaned_multimodal_dataset.csv')

# Extract verified audio-only numerical predictors
audio_features = [
    c for c in predictor_columns(df, modality='audio') 
    if df[c].notna().any() and df[c].nunique(dropna=True) > 1
]

print(f'Total Participant Observations: {len(df)}')
print(f'Total Acoustic Predictor Features: {len(audio_features)}')
print(f'Valence Class Distribution: {dict(df["valence_group"].value_counts())}')
print(f'Arousal Class Distribution: {dict(df["arousal_group"].value_counts())}')

## 3. Model Architecture & Pipeline Setup
We compare 4 machine learning models plus a Majority-Class Baseline:
1. **Majority Baseline (`DummyClassifier`)**
2. **Logistic Regression (L2 / Ridge, $C=0.1$)**
3. **Support Vector Machine (Linear kernel, $C=0.1$)**
4. **Support Vector Machine (RBF kernel, $C=1.0$)**
5. **Random Forest Classifier (100 trees, max depth 3)**

In [ ]:
X = df[audio_features].values
y_val = df['valence_group'].values
y_aro = df['arousal_group'].values

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Majority Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression (L2, C=0.1)': LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    'SVM (Linear, C=0.1)': SVC(kernel='linear', C=0.1, random_state=42),
    'SVM (RBF, C=1.0)': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'Random Forest (100 trees, depth 3)': RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
}

## 4. Experiment 1: 3-Class Valence Classification
* Classes: **Negative** ($n=13$), **Neutral** ($n=13$), **Positive** ($n=14$)
* Chance Level $\approx 33.3\%$

In [ ]:
val_results = []
val_preds = {}
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

for name, clf in models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', clf)
    ])
    cv_res = cross_validate(pipe, X, y_val, cv=cv5, scoring=scoring)
    y_pred = cross_val_predict(pipe, X, y_val, cv=cv5)
    val_preds[name] = y_pred

    val_results.append({
        'Model': name,
        'Accuracy': f"{cv_res['test_accuracy'].mean()*100:.1f}% (+/- {cv_res['test_accuracy'].std()*100:.1f}%)",
        'Precision': f"{cv_res['test_precision_macro'].mean():.3f}",
        'Recall': f"{cv_res['test_recall_macro'].mean():.3f}",
        'Macro F1': f"{cv_res['test_f1_macro'].mean():.3f} (+/- {cv_res['test_f1_macro'].std():.3f})"
    })

df_val_res = pd.DataFrame(val_results)
display(df_val_res)

## 5. Experiment 2: 3-Class Arousal Classification
* Classes: **Low** ($n=13$), **Moderate** ($n=16$), **High** ($n=11$)
* Majority Baseline: $40.0\%$

In [ ]:
aro_results = []
aro_preds = {}

for name, clf in models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', clf)
    ])
    cv_res = cross_validate(pipe, X, y_aro, cv=cv5, scoring=scoring)
    y_pred = cross_val_predict(pipe, X, y_aro, cv=cv5)
    aro_preds[name] = y_pred

    aro_results.append({
        'Model': name,
        'Accuracy': f"{cv_res['test_accuracy'].mean()*100:.1f}% (+/- {cv_res['test_accuracy'].std()*100:.1f}%)",
        'Precision': f"{cv_res['test_precision_macro'].mean():.3f}",
        'Recall': f"{cv_res['test_recall_macro'].mean():.3f}",
        'Macro F1': f"{cv_res['test_f1_macro'].mean():.3f} (+/- {cv_res['test_f1_macro'].std():.3f})"
    })

df_aro_res = pd.DataFrame(aro_results)
display(df_aro_res)

## 6. Confusion Matrices for Winning Models
Compare out-of-fold confusion matrices for the best performing models.

In [ ]:
cm_val = confusion_matrix(y_val, val_preds['Random Forest (100 trees, depth 3)'], labels=['Negative', 'Neutral', 'Positive'])
cm_aro = confusion_matrix(y_aro, aro_preds['SVM (Linear, C=0.1)'], labels=['Low', 'Moderate', 'High'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'], ax=axes[0])
axes[0].set_title('Valence 3-Class Confusion Matrix\n(Random Forest: 100 Trees, Depth 3)', fontweight='bold')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('True Ground Truth')

sns.heatmap(cm_aro, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Low', 'Moderate', 'High'],
            yticklabels=['Low', 'Moderate', 'High'], ax=axes[1])
axes[1].set_title('Arousal 3-Class Confusion Matrix\n(SVM Linear: C=0.1)', fontweight='bold')
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('True Ground Truth')

plt.tight_layout()
plt.show()

## 7. Conclusions & Findings
1. **Valence Ambiguity:** Random Forest achieves $32.5\%$ accuracy, reflecting that acoustic features alone cannot easily distinguish positive from negative valence in spontaneous classroom speech.
2. **Arousal Predictability:** Linear SVM achieves $47.5\%$ in 5-fold CV and $55.0\%$ in LOOCV, significantly exceeding the $40.0\%$ baseline. Pitch and vocal energy reliably track physiological activation.
3. **Cross-Validation Necessity:** Large fold variances ($\pm 6.1\%$ to $\pm 18.7\%$) demonstrate that cross-validation is essential for reliable reporting.